# Current results

One workbook, three sheets: Combined, Clinical, and Dscope. Clinical reads the photo retrieval run. A metric that is not on disk yet is a blank cell.

`IMAGE_ENCODER` picks the codebook embedding and `SCALER_DATASET` picks the balancing weight. `dinov2` with `coco` reads the current folders and writes `results/report_YYYYMMDD.xlsx`. Another encoder or `derm` reads the matching suffix, such as `standalone_dual_10_photo_panderm_derm`, and writes `results/report_{encoder}_{dataset}_YYYYMMDD.xlsx`. Each workbook still has the three sheets.

Higher is better in every column. Reliability, text generality, and image generality are successes on the edited cases. Locality is edited-set neighbors that nobody marked wrong and that keep the same top-1. OOD generality is leftover cases the saved prediction got wrong that become correct. OOD locality is leftover cases the saved prediction got right that stay correct. OOD accuracy is the same leftover cases against the 16-class label, before and after the codebook.

Re-run the next cell to refresh `results/report_YYYYMMDD.xlsx`.


In [1]:
import json
from datetime import date
from pathlib import Path

from IPython.display import Markdown, display
from openpyxl import Workbook

EMBED_MODE = "standalone_dual"
REJECT_THRESHOLD = 10
IMAGE_ENCODER = "dinov2"
SCALER_DATASET = "coco"
MODES = (("combined", "Combined"), ("photo", "Clinical"), ("dscope", "Dscope"))

ROOT = Path("/scratch/jq2uw/MME/derm_vlm_edit/results")
MODELS = (("medgemma", "MedGemma"), ("dermatollama", "DermatoLlama"))


def load(folder, metric, model):
    path = folder / f"{metric}__all__{model}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text())


def pair(payload, num, den):
    if not payload or payload.get(num) is None or payload.get(den) is None:
        return None
    return f"{payload[num]}/{payload[den]}"


def kept(payload):
    if not payload:
        return None
    if payload.get("n_kept") is not None:
        return payload["n_kept"]
    if payload.get("n_orig_correct") is None or payload.get("n_broken") is None:
        return None
    return payload["n_orig_correct"] - payload["n_broken"]


def rows_for(mode):
    tag = f"{EMBED_MODE}_{float(REJECT_THRESHOLD):g}"
    if mode != "combined":
        tag = f"{tag}_{mode}"
    encoder = IMAGE_ENCODER.strip().lower()
    if encoder != "dinov2":
        tag = f"{tag}_{encoder}"
    dataset = SCALER_DATASET.strip().lower()
    if dataset != "coco":
        tag = f"{tag}_{dataset}"
    perturb = ROOT / f"perturb_gen/{tag}/collect"
    ood = ROOT / f"ood_gen/{tag}/collect"
    rows = []
    for short, name in MODELS:
        reliability = load(perturb, "reliability", short)
        text = load(perturb, "text_generality", short)
        image = load(perturb, "image_generality", short)
        locality = load(perturb, "locality", short)
        generality = load(ood, "generality", short)
        stay = None if generality is None else dict(generality, n_kept=kept(generality))
        rows.append([
            name,
            pair(reliability, "n_correct", "n"),
            pair(text, "n_correct", "n"),
            pair(image, "n_correct", "n"),
            pair(locality, "n_correct", "n"),
            pair(generality, "n_fixed", "n_orig_wrong"),
            pair(stay, "n_kept", "n_orig_correct"),
            pair(generality, "n_correct_before", "n"),
            pair(generality, "n_correct", "n"),
        ])
    return rows


header = [
    "Model", "Reliability", "Text generality", "Image generality", "Locality",
    "OOD generality", "OOD locality", "OOD accuracy before", "OOD accuracy after",
]
definitions = [
    ["Column", "Definition"],
    ["Reliability", "On cases a dermatologist marked wrong and we edited, how often the model top-1 matches the human correction."],
    ["Text generality", "Same edited cases, but with a rephrased question. How often the correction still holds."],
    ["Image generality", "Same edited cases, but with a slightly changed image. How often the correction still holds."],
    ["Locality", "On cases nobody marked wrong, how often the top-1 stays the same after retrieval. A lower number means more originally correct answers were flipped."],
    ["OOD generality", "On leftover MIDAS cases dermatologists did not annotate, among originally wrong predictions, how many become correct."],
    ["OOD locality", "On those leftover cases, among originally correct predictions, how many stay correct."],
    ["OOD accuracy before", "Leftover cases, unedited model top-1 vs the 16-class label."],
    ["OOD accuracy after", "Leftover cases, edited model top-1 vs the 16-class label."],
]

book = Workbook()
blocks = []
for i, (mode, title) in enumerate(MODES):
    sheet = book.active if i == 0 else book.create_sheet()
    sheet.title = title
    table = [header, *rows_for(mode), [], *definitions]
    for row in table:
        sheet.append([None if cell in ("", None) else cell for cell in row])
    md = [f"### {title}", "", "| " + " | ".join(header) + " |", "| " + " | ".join("---" for _ in header) + " |"]
    md.extend("| " + " | ".join("" if cell is None else str(cell) for cell in row) + " |" for row in table[1:3])
    blocks.append("\n".join(md))

encoder = IMAGE_ENCODER.strip().lower()
dataset = SCALER_DATASET.strip().lower()
stamp = f"{date.today():%Y%m%d}"
name = f"report_{stamp}.xlsx" if encoder == "dinov2" and dataset == "coco" else f"report_{encoder}_{dataset}_{stamp}.xlsx"
out = ROOT / name
book.save(out)
print("saved", out)
display(Markdown("\n\n".join(blocks)))


saved /scratch/jq2uw/MME/derm_vlm_edit/results/report_20261122.xlsx


### Combined

| Model | Reliability | Text generality | Image generality | Locality | OOD generality | OOD locality | OOD accuracy before | OOD accuracy after |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MedGemma | 50/50 | 41/50 | 50/50 | 34/171 | 55/483 | 51/263 | 263/746 | 106/746 |
| DermatoLlama | 30/30 | 24/30 | 30/30 | 64/184 | 49/480 | 81/263 | 263/743 | 130/743 |

### Clinical

| Model | Reliability | Text generality | Image generality | Locality | OOD generality | OOD locality | OOD accuracy before | OOD accuracy after |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MedGemma | 50/50 | 48/50 | 49/50 | 49/171 | 36/483 | 88/263 | 263/746 | 124/746 |
| DermatoLlama | 30/30 | 29/30 | 30/30 | 68/184 | 40/480 | 86/263 | 263/743 | 126/743 |

### Dscope

| Model | Reliability | Text generality | Image generality | Locality | OOD generality | OOD locality | OOD accuracy before | OOD accuracy after |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MedGemma | 50/50 | 50/50 | 50/50 | 35/171 | 47/483 | 50/263 | 263/746 | 97/746 |
| DermatoLlama | 30/30 | 24/30 | 30/30 | 89/184 | 40/480 | 117/263 | 263/743 | 157/743 |